# De la Emoción a la Ecuación · Jueves 1
## Captura y Alfa

**Castle Quant · La ciencia del alfa.** Método CASTLE: hoy **C · Captura** y **A · Alfa**.

| # | Sección | Qué sale |
|---|---|---|
| 1 | Preparación | Las librerías y el estilo de las gráficas |
| 2 | Parámetros | Los números que definen la regla |
| 3 | Captura | `velas`: una fila por día con apertura, máximo, mínimo y cierre |
| 4 | Limpieza | `velas` sin duplicados, sin precios vacíos y sin la vela de hoy |
| 5 | Partición | `velas` queda solo con el entrenamiento; la prueba se guarda sin mirarla |
| 6 | Retornos | `retornos`: cuánto ganó o perdió Bitcoin cada día |
| 7 | Volatilidad | `atr`: cuánto recorre el precio en un día normal |
| 8 | Señal | `regla`: el canal y la señal de cada día (+1 largo, −1 corto, 0 nada) |
| 9 | Plan | `planes`: entrada, stop y objetivo de cada señal |
| 10 | Reto en vivo | Una prueba que detecta si un cálculo mira el futuro |
| 11 | Reto de la semana | La misma señal con otro canal |
| 12 | Guardar | Los dos archivos para el jueves 2 |

> Contenido educativo; no es recomendación de inversión. Los resultados históricos no garantizan resultados futuros.

## 1 · Preparación

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

# Colores de Castle Quant para las gráficas
NAVY, LINEA, TEXTO, GRIS, AZUL, DORADO = "#0b1a2e", "#22395a", "#f5f7fa", "#8e9ab5", "#7fa3e6", "#cfb97d"
plt.rcParams.update({
    "figure.facecolor": NAVY, "axes.facecolor": NAVY, "axes.edgecolor": LINEA, "axes.labelcolor": TEXTO,
    "text.color": TEXTO, "xtick.color": GRIS, "ytick.color": GRIS, "grid.color": LINEA, "axes.grid": True,
    "axes.titleweight": "bold", "figure.figsize": (12, 5), "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False,
})

## 2 · Parámetros

La única celda que se cambia.

In [ ]:
ACTIVO = "BTC-USD"            # Bitcoin contra dólar en Yahoo Finance
FECHA_INICIO = "2014-09-17"   # primer día con datos
FECHA_CORTE = "2022-12-31"    # último día de entrenamiento; lo posterior es prueba y hoy no se mira

CANAL = 20                    # días del canal de ruptura
DIAS_ATR = 20                 # días con los que se mide el ATR
ATR_STOP = 2.0                # el stop va a 2 ATR de la entrada
ATR_OBJETIVO = 4.0            # el objetivo va a 4 ATR de la entrada

ARCHIVO_ENTRENAMIENTO = "jueves1_entrenamiento.csv"
ARCHIVO_PRUEBA = "jueves1_prueba.csv"

---
# C · Captura

## 3 · Captura

**Entra:** `ACTIVO`, `FECHA_INICIO` · **Sale:** `velas`

In [ ]:
datos = yf.download(ACTIVO, start=FECHA_INICIO, interval="1d", auto_adjust=True, progress=False)
datos.columns = datos.columns.get_level_values(0)     # yfinance entrega columnas en dos niveles: se deja uno

velas = datos.rename(columns={"Open": "apertura", "High": "maximo", "Low": "minimo", "Close": "cierre"})
velas = velas[["apertura", "maximo", "minimo", "cierre"]].rename_axis(columns=None)
velas.index = velas.index.tz_localize(None).rename("Fecha")

velas.tail()

## 4 · Limpieza

**Entra:** `velas` · **Sale:** `velas` sin duplicados, sin precios vacíos y sin la vela de hoy

In [ ]:
antes = len(velas)

velas = velas.sort_index()
velas = velas[~velas.index.duplicated(keep="last")]     # una sola vela por fecha
velas = velas[(velas > 0).all(axis=1)]                  # sin precios vacíos ni negativos

hoy = pd.Timestamp.now(tz="UTC").tz_localize(None).normalize()
velas = velas[velas.index < hoy]                        # la vela de hoy aún no cierra (cierra a las 00:00 UTC)

print(f"Velas: {antes:,} descargadas · {len(velas):,} limpias, del {velas.index[0]:%Y-%m-%d} al {velas.index[-1]:%Y-%m-%d}")

## 5 · Partición

**Entra:** `velas`, `FECHA_CORTE` · **Sale:** `velas` solo con el entrenamiento, y el archivo de prueba guardado sin mirarlo

In [ ]:
corte = pd.Timestamp(FECHA_CORTE)
total, fin_de_los_datos = len(velas), velas.index[-1]

prueba = velas[velas.index > corte]          # lo posterior al corte: no se grafica ni se resume
velas = velas[velas.index <= corte]          # desde aquí, `velas` es solo el entrenamiento

prueba.to_csv(ARCHIVO_PRUEBA)                # se guarda sellada para el final del jueves 2
print(f"Entrenamiento: {len(velas):,} velas ({len(velas) / total:.0%}), hasta {velas.index[-1]:%Y-%m-%d}")
print(f"Prueba:        {len(prueba):,} velas ({len(prueba) / total:.0%}), guardadas en {ARCHIVO_PRUEBA}")
del prueba, datos

In [ ]:
fig, ax = plt.subplots()
ax.plot(velas["cierre"], color=AZUL, lw=1.4, label="Entrenamiento")
ax.axvspan(corte, fin_de_los_datos, color=GRIS, alpha=0.18, label="Prueba: sellada hasta el jueves 2")
ax.set_yscale("log")
ax.set_title(f"{ACTIVO} · cierre diario (escala logarítmica)")
ax.legend(loc="upper left")
plt.show()

## 6 · Retornos

**Entra:** `velas` · **Sale:** `retornos`

$$r_t = \frac{C_t}{C_{t-1}} - 1$$

In [ ]:
retornos = velas["cierre"].pct_change().dropna()
media, desviacion = retornos.mean(), retornos.std()

pd.Series({
    "Retorno medio diario": f"{media:.3%}",
    "Desviación estándar diaria": f"{desviacion:.3%}",
    "Volatilidad anual (× √365)": f"{desviacion * math.sqrt(365):.1%}",
    "Días positivos": f"{(retornos > 0).mean():.1%}",
    "Mejor día": f"{retornos.max():.2%} ({retornos.idxmax():%Y-%m-%d})",
    "Peor día": f"{retornos.min():.2%} ({retornos.idxmin():%Y-%m-%d})",
}).to_frame(ACTIVO)

In [ ]:
# Histograma de los retornos contra una campana normal con la misma media y desviación
x = np.linspace(retornos.quantile(0.001), retornos.quantile(0.999), 400)
campana = np.exp(-0.5 * ((x - media) / desviacion) ** 2) / (desviacion * math.sqrt(2 * math.pi))

fig, ax = plt.subplots()
ax.hist(retornos, bins=150, density=True, color=AZUL, alpha=0.75, label="Retornos diarios")
ax.plot(x, campana, color=DORADO, lw=2.5, label="Distribución normal")
ax.set_xlim(x[0], x[-1])
ax.set_title("Distribución de los retornos diarios")
ax.legend()
plt.show()

extremos = int((abs(retornos - media) > 4 * desviacion).sum())
esperados = len(retornos) * math.erfc(4 / math.sqrt(2))       # P(|Z| > 4) en una normal
print(f"Días a más de 4 desviaciones: {extremos} reales contra {esperados:.2f} esperados en una normal")

## 7 · Volatilidad

**Entra:** `velas`, `DIAS_ATR` · **Sale:** `atr`

$$TR_t = \max\big(H_t - L_t,\; |H_t - C_{t-1}|,\; |L_t - C_{t-1}|\big) \qquad ATR_t = \text{promedio de los últimos } n \text{ valores de } TR$$

In [ ]:
def calcular_atr(velas, dias):
    """ATR: promedio del rango verdadero de los últimos `dias` días."""
    cierre_previo = velas["cierre"].shift(1)
    rango_verdadero = pd.concat([
        velas["maximo"] - velas["minimo"],               # recorrido del día
        (velas["maximo"] - cierre_previo).abs(),         # salto hacia arriba desde el cierre de ayer
        (velas["minimo"] - cierre_previo).abs(),         # salto hacia abajo desde el cierre de ayer
    ], axis=1).max(axis=1, skipna=False)
    return rango_verdadero.rolling(dias).mean()


atr = calcular_atr(velas, DIAS_ATR)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(atr / velas["cierre"], color=DORADO, lw=1.3)
ax.yaxis.set_major_formatter(lambda valor, _: f"{valor:.0%}")
ax.set_title(f"ATR de {DIAS_ATR} días, como porcentaje del precio")
plt.show()

print(f"Último día: 1 ATR = {atr.iloc[-1]:,.0f} USD, el {atr.iloc[-1] / velas['cierre'].iloc[-1]:.1%} del precio")

---
# A · Alfa

## 8 · Señal

**Entra:** `velas`, `CANAL` · **Sale:** `regla`, con el canal y la señal de cada día

$$s_t = \begin{cases} +1 & \text{si el cierre supera el máximo de los } N \text{ días anteriores}\\ -1 & \text{si el cierre cae bajo el mínimo de los } N \text{ días anteriores}\\ \;\;\,0 & \text{si no rompe}\end{cases}$$

In [ ]:
def senal_de_ruptura(velas, canal):
    """Devuelve el canal de los días anteriores y la señal al cierre: +1 largo, -1 corto, 0 nada."""
    canal_alto = velas["maximo"].rolling(canal).max().shift(1)    # shift(1): el canal no incluye la vela de hoy
    canal_bajo = velas["minimo"].rolling(canal).min().shift(1)

    senal = pd.Series(0.0, index=velas.index)
    senal[velas["cierre"] > canal_alto] = 1.0                     # rompe hacia arriba: largo
    senal[velas["cierre"] < canal_bajo] = -1.0                    # rompe hacia abajo: corto
    senal = senal.where(canal_alto.notna())                       # sin canal completo no hay señal
    return pd.DataFrame({"canal_alto": canal_alto, "canal_bajo": canal_bajo, "senal": senal})


regla = senal_de_ruptura(velas, CANAL)

anios = len(regla.dropna()) / 365
largas, cortas = int((regla["senal"] == 1).sum()), int((regla["senal"] == -1).sum())
print(f"Días con señal: {largas} de largo y {cortas} de corto · unos {(largas + cortas) / anios:.0f} por año")

In [ ]:
# El último año: precio, canal y señales
tramo = pd.concat([velas["cierre"], regla], axis=1).iloc[-365:]
largas, cortas = tramo[tramo["senal"] == 1], tramo[tramo["senal"] == -1]

fig, ax = plt.subplots()
ax.plot(tramo["cierre"], color=AZUL, lw=1.4, label="Cierre")
ax.fill_between(tramo.index, tramo["canal_bajo"], tramo["canal_alto"], color=GRIS, alpha=0.15, label=f"Canal de {CANAL} días")
ax.scatter(largas.index, largas["cierre"], marker="^", s=55, color=DORADO, zorder=3, label="Señal de largo")
ax.scatter(cortas.index, cortas["cierre"], marker="v", s=55, color=TEXTO, zorder=3, label="Señal de corto")
ax.set_title("El último año: el precio, su canal y las señales")
ax.legend(loc="upper left")
plt.show()

## 9 · Plan

**Entra:** `regla`, `atr`, `ATR_STOP`, `ATR_OBJETIVO` · **Sale:** `planes`, con la entrada, el stop y el objetivo de cada señal

$$\text{stop} = \text{entrada} \mp 2 \cdot ATR \qquad\qquad \text{objetivo} = \text{entrada} \pm 4 \cdot ATR$$

In [ ]:
dias_con_senal = regla.index[regla["senal"].fillna(0) != 0]
lado = regla.loc[dias_con_senal, "senal"]                 # +1 largo, -1 corto
entrada = velas.loc[dias_con_senal, "cierre"]             # referencia: el cierre del día de la señal
distancia = atr.loc[dias_con_senal]                       # 1 ATR de ese día

planes = pd.DataFrame({
    "lado": lado.map({1.0: "largo", -1.0: "corto"}),
    "entrada": entrada,
    "stop": entrada - lado * ATR_STOP * distancia,          # en contra: si llega ahí, se cierra con pérdida
    "objetivo": entrada + lado * ATR_OBJETIVO * distancia,  # a favor: si llega ahí, se toma la ganancia
})
planes.tail()

In [ ]:
# El plan de la última señal, sobre el precio
plan, dia = planes.iloc[-1], planes.index[-1]
tramo = velas["cierre"].iloc[-120:]
fin = tramo.index[-1] + pd.Timedelta(days=15)

fig, ax = plt.subplots()
ax.plot(tramo, color=AZUL, lw=1.5, label="Cierre")
for nivel, color, estilo in [("entrada", TEXTO, "-"), ("stop", GRIS, "--"), ("objetivo", DORADO, "--")]:
    ax.hlines(plan[nivel], dia, fin, color=color, ls=estilo, lw=1.8, label=f"{nivel.capitalize()}: {plan[nivel]:,.0f}")
ax.set_xlim(tramo.index[0], fin)
ax.set_title(f"Plan de la señal del {dia:%Y-%m-%d}: {plan['lado']}")
ax.legend(loc="upper left")
plt.show()

print(f"Se arriesga 1 para buscar {ATR_OBJETIVO / ATR_STOP:g}: la regla empata si acierta el {ATR_STOP / (ATR_STOP + ATR_OBJETIVO):.1%} de las veces.")

---
## 10 · Reto en vivo: ¿cuál de estas tres líneas mira el futuro?

```python
A)  canal = maximo.rolling(20).max().shift(1)
B)  canal = maximo.rolling(20, center=True).max()
C)  canal = maximo.expanding().max().shift(1)
```

**Responde en tu cuenta gratis de Castle Quant antes de ejecutar la celda.**

In [ ]:
def mira_el_futuro(calcular, datos):
    """True si los valores pasados de un cálculo cambian cuando se le quitan los datos futuros."""
    corte = datos.index[len(datos) // 2]
    con_todo = calcular(datos).loc[:corte]           # calculado con toda la historia
    sin_futuro = calcular(datos.loc[:corte])         # calculado solo con lo anterior al corte
    return not con_todo.equals(sin_futuro)


opciones = {
    "A · rolling(20).max().shift(1)": lambda maximo: maximo.rolling(20).max().shift(1),
    "B · rolling(20, center=True).max()": lambda maximo: maximo.rolling(20, center=True).max(),
    "C · expanding().max().shift(1)": lambda maximo: maximo.expanding().max().shift(1),
}
for nombre, calcular in opciones.items():
    print(f"{nombre:<36} {'MIRA EL FUTURO' if mira_el_futuro(calcular, velas['maximo']) else 'solo usa el pasado'}")

# La misma prueba, aplicada a la regla de este notebook
nuestra = mira_el_futuro(lambda v: senal_de_ruptura(v, CANAL)["senal"], velas)
print(f"{'Nuestra señal de ruptura':<36} {'MIRA EL FUTURO' if nuestra else 'solo usa el pasado'}")

## 11 · Reto de la semana

Cambia `CANAL_RETO` a **10** y a **55**. ¿Cuántos días con señal por año da cada uno, y cómo se reparten entre largos y cortos? Sube tu captura a la comunidad de Castle Quant.

In [ ]:
CANAL_RETO = 10   # cámbialo a 55 y vuelve a ejecutar

senal_reto = senal_de_ruptura(velas, CANAL_RETO)["senal"].dropna()
largas, cortas = int((senal_reto == 1).sum()), int((senal_reto == -1).sum())
print(f"Canal de {CANAL_RETO} días: {largas} señales de largo y {cortas} de corto · unos {(largas + cortas) / (len(senal_reto) / 365):.0f} por año")

## 12 · Guardar

**Entra:** `velas`, `atr`, `regla` · **Sale:** `jueves1_entrenamiento.csv`. El archivo de prueba ya quedó guardado en la sección 5 y no se abre hasta el final del jueves 2.

In [ ]:
entrenamiento = pd.concat([velas, atr.rename("atr"), regla], axis=1)
entrenamiento.to_csv(ARCHIVO_ENTRENAMIENTO)

print(f"Guardado: {ARCHIVO_ENTRENAMIENTO} · {len(entrenamiento):,} filas · columnas: {', '.join(entrenamiento.columns)}")
print(f"Guardado en la sección 5, sin abrir: {ARCHIVO_PRUEBA}")

---
**El próximo jueves:** Simulación, Tamaño y Libre de sesgos. Lleva los dos archivos.

Comunidad Castle Quant en WhatsApp: https://chat.whatsapp.com/LfKXZoL6xjRF32c6ZMBKzc